<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.3-batch-routing/notebooks/GCP_Capstone_10.3_BatchRouting.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.3 Batch API + Model Routing — 50% + Tier Savings
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-genai google-cloud-storage google-cloud-bigquery

from google import genai
from google.genai import types
from google.genai.types import CreateBatchJobConfig, GenerateContentConfig
import json, time, os

PROJECT = 'your-project-id'
LOCATION = 'us-central1'
BUCKET = 'documind-batch'

client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)
print('SDK ready')


## Cell 1: Build Batch JSONL


In [ ]:
# Create JSONL with proper batch format
# Vertex batch reads ONLY the 'request' key (a GenerateContentRequest in camelCase:
# generationConfig / maxOutputTokens). It ignores extra top-level fields and
# correlates outputs to inputs by ROW ORDER (each output line = your request + a response).
# We keep an 'id' here only as a local bookkeeping convenience; do NOT rely on Vertex
# echoing it back -- preserve input ordering instead.
def build_batch_jsonl(documents: list[dict], output_path: str):
    '''documents: [{'id': '...', 'prompt': '...'}, ...]'''
    with open(output_path, 'w') as f:
        for doc in documents:
            record = {
                'id': doc['id'],  # local-only; Vertex ignores non-'request' keys
                'request': {
                    'contents': [{
                        'role': 'user',
                        'parts': [{'text': doc['prompt']}]
                    }],
                    'generationConfig': {
                        'temperature': 0.2,
                        'maxOutputTokens': 200,
                    }
                }
            }
            f.write(json.dumps(record) + '\n')
    return output_path

# Sample DocuMind classification batch
docs = [
    {'id': f'doc-{i:03d}',
     'prompt': f'Classify as INVOICE/CONTRACT/REPORT: Sample doc {i} content...'}
    for i in range(10)
]
build_batch_jsonl(docs, 'classify-batch.jsonl')

# Validate parsing
with open('classify-batch.jsonl') as f:
    count = 0
    for line in f:
        json.loads(line)  # raises on syntax error
        count += 1
print(f'OK: {count} valid records in classify-batch.jsonl')


## Cell 2: Submit Batch Job (template)


In [ ]:
# Template - requires real GCS bucket
def launch_batch_job(model: str, gcs_input: str, gcs_output: str, display_name: str):
    '''Submit batch job from GCS JSONL to GCS output directory.'''
    job = client.batches.create(
        model=model,
        src=gcs_input,
        config=CreateBatchJobConfig(
            dest=gcs_output,
            display_name=display_name
        ),
    )
    return job

def poll_until_complete(job, interval: int = 30):
    TERMINAL = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED',
                'JOB_STATE_CANCELLED', 'JOB_STATE_PAUSED'}
    while str(job.state) not in TERMINAL:
        print(f'  State: {job.state}')
        time.sleep(interval)
        job = client.batches.get(name=job.name)
    print(f'Final: {job.state}')
    return job

print('Ready to submit batch jobs.')
print('Example call (requires real GCS URIs):')
print('  job = launch_batch_job("gemini-3.6-flash", "gs://bucket/in.jsonl", "gs://bucket/out/", "nightly")')
print('  job = poll_until_complete(job)')


## Cell 3: Rule-Based Tier Router


In [ ]:
# Deterministic routing - zero latency overhead
COMPLEX_SIGNALS = ['analyze', 'compare', 'evaluate', 'synthesize',
                    'why does', 'implications', 'reasoning']
SIMPLE_SIGNALS = ['classify', 'extract', 'what is', 'list', 'translate',
                   'find', 'name', 'identify']

def rule_route(query: str, doc_length: int = 0) -> str:
    '''Map query + doc_length to cheapest capable model.'''
    q = query.lower()
    has_simple = any(s in q for s in SIMPLE_SIGNALS)
    has_complex = any(s in q for s in COMPLEX_SIGNALS)
    
    # Override: very long docs need Pro's larger effective context handling
    if doc_length > 50000:
        return 'gemini-3.1-pro-preview'
    # Complex reasoning signals
    if has_complex:
        return 'gemini-3.1-pro-preview'
    # Simple extraction/classification on short docs
    if has_simple and doc_length < 5000:
        return 'gemini-3.1-flash-lite'
    # Default: Flash handles everything else
    return 'gemini-3.6-flash'

# Test routing on sample queries
tests = [
    ('Classify this invoice', 500),
    ('Summarize Q3 earnings', 8000),
    ('Analyze the strategic implications of the merger', 12000),
    ('Extract vendor name', 300),
    ('Compare compliance risks across these three contracts', 75000),
]
print(f'{"Query":<55} {"DocLen":>7}  Model')
print('-' * 85)
for q, dl in tests:
    m = rule_route(q, dl)
    print(f'{q:<55} {dl:>7}  {m}')


## Cell 4: LLM-Based Router (Flash-Lite Classifier)


In [ ]:
# Uses Flash-Lite to classify, ~$0.000001 per routing call
def llm_route(query: str) -> tuple[str, str]:
    '''Returns (tier_label, model_name).'''
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=query,
        config=GenerateContentConfig(
            system_instruction=(
                'Classify query complexity. Reply with ONE word only:\n'
                'SIMPLE  - factual lookup, classification, extraction, formatting\n'
                'MEDIUM  - summarization, Q&A, moderate analysis\n'
                'COMPLEX - multi-step reasoning, nuanced analysis, synthesis'
            ),
            temperature=0,
            max_output_tokens=10,
        ),
    )
    tier = response.text.strip().upper()
    model_map = {
        'SIMPLE':  'gemini-3.1-flash-lite',
        'MEDIUM':  'gemini-3.6-flash',
        'COMPLEX': 'gemini-3.1-pro-preview',
    }
    return tier, model_map.get(tier, 'gemini-3.6-flash')

# Test (template - requires actual API call)
print('LLM-based routing function ready')
print('Example: tier, model = llm_route("Summarize this contract")')
print('  ~10 tokens routing cost = $0.000001 per query (negligible)')


## Cell 5: Pricing & Savings Calculator


In [ ]:
# April 2026 pricing per 1M tokens (Vertex AI)
PRICING = {
    'gemini-3.1-flash-lite':  {'in_std': 0.25, 'in_batch': 0.125,
                                'out_std': 1.50, 'out_batch': 0.75},
    'gemini-3.6-flash':       {'in_std': 1.50, 'in_batch': 0.75,
                                'out_std': 7.50, 'out_batch': 3.75},
    'gemini-3.1-pro-preview':         {'in_std': 2.00, 'in_batch': 1.00,
                                'out_std': 12.00, 'out_batch': 6.00},
}

# tier -> current 3.x model id (keys must match PRICING)
TIER_MODEL = {'flash-lite': 'gemini-3.1-flash-lite', 'flash': 'gemini-3.6-flash', 'pro': 'gemini-3.1-pro-preview'}

def calc_cost(model: str, input_tokens: int, output_tokens: int, batch: bool = False) -> float:
    p = PRICING[model]
    in_rate = p['in_batch' if batch else 'in_std']
    out_rate = p['out_batch' if batch else 'out_std']
    return (input_tokens * in_rate + output_tokens * out_rate) / 1_000_000

def documind_scenario(n_docs: int, avg_in: int, avg_out: int,
                       distribution: dict = None):
    '''distribution: {'flash-lite': 0.65, 'flash': 0.25, 'pro': 0.10}'''
    if distribution is None:
        distribution = {'flash-lite': 0.65, 'flash': 0.25, 'pro': 0.10}
    
    # Baseline: all Pro standard
    all_pro_std = calc_cost('gemini-3.1-pro-preview', n_docs * avg_in, n_docs * avg_out)
    
    # All Pro batch (50% off)
    all_pro_batch = calc_cost('gemini-3.1-pro-preview', n_docs * avg_in, n_docs * avg_out, batch=True)
    
    # Routed standard
    routed_std = 0
    for tier, pct in distribution.items():
        model = TIER_MODEL[tier]
        docs = int(n_docs * pct)
        routed_std += calc_cost(model, docs * avg_in, docs * avg_out)
    
    # Routed + batch (final winner)
    routed_batch = 0
    for tier, pct in distribution.items():
        model = TIER_MODEL[tier]
        docs = int(n_docs * pct)
        routed_batch += calc_cost(model, docs * avg_in, docs * avg_out, batch=True)
    
    return {
        'all_pro_standard': round(all_pro_std, 2),
        'all_pro_batch':    round(all_pro_batch, 2),
        'routed_standard':  round(routed_std, 2),
        'routed_batch':     round(routed_batch, 2),
        'savings_vs_base':  round((1 - routed_batch / all_pro_std) * 100, 1),
    }

# DocuMind monthly: 100K documents, 2000 input + 500 output tokens
r = documind_scenario(100_000, 2000, 500)
print('DocuMind monthly cost (100K documents, avg 2000/500 tokens):')
print(f'  All Pro standard:  ${r["all_pro_standard"]:>8.2f}')
print(f'  All Pro batch:     ${r["all_pro_batch"]:>8.2f}')
print(f'  Routed standard:   ${r["routed_standard"]:>8.2f}')
print(f'  Routed + batch:    ${r["routed_batch"]:>8.2f}  <-- WINNER')
print(f'  Savings vs baseline: {r["savings_vs_base"]}%')


## Cell 6: Batch + Routing Pipeline


In [ ]:
# End-to-end: classify each doc, partition into per-tier JSONL, launch 3 batch jobs
def batch_with_routing(documents: list[dict], use_gcs: bool = False):
    '''documents: [{'id', 'prompt', 'doc_length'}, ...]'''
    tiers = {'flash-lite': [], 'flash': [], 'pro': []}
    
    # Step 1: Classify each doc into a tier
    for doc in documents:
        model = rule_route(doc['prompt'], doc.get('doc_length', 0))
        # Extract tier name from model name
        if 'lite' in model: tiers['flash-lite'].append(doc)
        elif 'pro' in model: tiers['pro'].append(doc)
        else: tiers['flash'].append(doc)
    
    # Step 2: Write per-tier JSONL
    files = {}
    for tier, items in tiers.items():
        if not items: continue
        path = f'{tier}-batch.jsonl'
        build_batch_jsonl(items, path)
        files[tier] = path
    
    # Step 3: Submit batch jobs (template - needs GCS upload)
    model_map = {
        'flash-lite': 'gemini-3.1-flash-lite',
        'flash': 'gemini-3.6-flash',
        'pro': 'gemini-3.1-pro-preview',
    }
    
    stats = {}
    for tier, path in files.items():
        stats[tier] = {
            'count': len(tiers[tier]),
            'file': path,
            'model': model_map[tier],
        }
    return stats

# Test with 100 sample docs
sample_docs = [
    {'id': f'd{i}', 'prompt': rq, 'doc_length': dl}
    for i, (rq, dl) in enumerate([
        ('Classify as invoice or receipt', 400) for _ in range(60)
    ] + [
        ('Summarize this Q3 report', 8000) for _ in range(25)
    ] + [
        ('Analyze strategic implications', 60000) for _ in range(15)
    ])
]

stats = batch_with_routing(sample_docs)
print('Routing distribution for 100 docs:')
for tier, info in stats.items():
    print(f'  {tier:>10}: {info["count"]:>3} docs -> {info["model"]}')


## Cell 7: Collect Failures from Batch Output


In [ ]:
# Parse batch output JSONL, extract failures for retry
def collect_failures(output_jsonl_path: str) -> list[dict]:
    '''Returns list of failed request bodies for resubmission.'''
    failures = []
    with open(output_jsonl_path) as f:
        for line in f:
            result = json.loads(line)
            # Batch output schema varies; check multiple fields
            status = result.get('status', '')
            error = result.get('error', {})
            
            if error or 'error' in str(status).lower():
                failures.append({
                    'id': result.get('id', 'unknown'),
                    'request': result.get('request', {}),
                    'error': error or status
                })
    return failures

def escalate_retry(failures: list[dict], to_model: str = 'gemini-3.6-flash'):
    '''Retry failures with a more capable model.'''
    retry_jsonl = [
        {'id': f['id'], 'request': f['request']}
        for f in failures
    ]
    # Write to temp file
    with open('retry.jsonl', 'w') as f:
        for r in retry_jsonl:
            f.write(json.dumps(r) + '\n')
    print(f'Retry file written: retry.jsonl ({len(retry_jsonl)} items)')
    print(f'Submit with: client.batches.create(model="{to_model}", src=...)')

print('Failure collection + escalation pattern ready')


## Cell 8: Per-Tenant Cost Attribution


In [ ]:
# Add labels to every request for billing attribution
def tenant_aware_generate(tenant_id: str, query: str, feature: str = 'qa'):
    '''Generate response with tenant billing labels.'''
    model = rule_route(query, len(query))
    tier = model.split('-')[-1]  # e.g., 'flash-lite'
    
    response = client.models.generate_content(
        model=model,
        contents=query,
        config=GenerateContentConfig(
            labels={
                'tenant':  tenant_id,
                'feature': feature,
                'tier':    tier,
                'env':     'production',
            }
        ),
    )
    return response

# Query template for per-tenant costs via BigQuery billing export
QUERY_TENANT_COSTS = '''
SELECT
  labels.value AS tenant,
  ROUND(SUM(cost), 2) AS monthly_cost_usd,
  COUNT(*) AS request_count
FROM `{project}.{dataset}.gcp_billing_export_v1_{billing_account_id}`,
  UNNEST(labels) AS labels
WHERE labels.key = 'tenant'
  AND service.description = 'Vertex AI'
  AND sku.description LIKE '%Gemini%'
  AND invoice.month = @invoice_month
GROUP BY tenant
ORDER BY monthly_cost_usd DESC
'''

print('Tenant-aware generation function ready')
print('Labels attached: tenant, feature, tier, env')
print('Billing export query template defined')


## Done!
- Batch JSONL construction with id+request fields
- client.batches.create() GCS and BigQuery patterns
- Rule-based tier router (zero latency)
- LLM-based tier router (Flash-Lite classifier)
- Pricing calculator with stacked optimization scenarios
- Full batch+routing pipeline (classify→partition→submit)
- Failure collection and escalation
- Per-tenant cost attribution via labels + BigQuery billing export
